In [1]:
from dotenv import load_dotenv
load_dotenv()


import requests
from minsearch import Index

from openai import OpenAI
openai_client = OpenAI()

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
print('Q1:', len(documents) )

Q1: 72


In [5]:
documents[:2]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [6]:
def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index 

In [7]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()


class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        course='llm-zoomcamp',
        model='gpt-5.4-mini'
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.course = course
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        boost_dict = {'content': 2.0, }
        #filter_dict = {'course': self.course}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
            #filter_dict=filter_dict
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc['filename'])
            lines.append('content: ' + doc['content'])
            lines.append('')

        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer

In [8]:
index = build_index(documents)

In [9]:
question = 'How does the agentic loop keep calling the model until it stops?'

assistant = RAGBase(index=index, llm_client=openai_client)
search_results = assistant.search(question)
print('Q2:', search_results[0]['filename'] ) 

Q2: 01-agentic-rag/lessons/14-agentic-loop.md


In [10]:
assistant = RAGBase(index=index, llm_client=openai_client)

In [11]:
question = 'How does the agentic loop keep calling the model until it stops?'
answer = assistant.rag(question)
answer

Response(id='resp_0ce321929f6008ae006a3289af087081968bc71c5e8e7d70fb', created_at=1781696943.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0ce321929f6008ae006a3289b1afb08196be45cee3112c320f', content=[ResponseOutputText(annotations=[], text='It keeps a `while True` loop around the model call.\n\nAfter each response, the code checks whether the model returned any `function_call` items:\n\n- if yes, it runs the tool, appends the tool output to `messages`, and loops again\n- if no, it breaks out of the loop and stops\n\nSo the stop condition is: **no function calls in the latest response**.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=1781696946.0, conversation=None, max_output_tokens=None, m

In [12]:
print('Q3:', answer.usage.input_tokens)

Q3: 7121


In [13]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

answer = calculate_gpt54mini_price(answer.usage.input_tokens, answer.usage.output_tokens)
print("Total cost: $", round(answer["total_cost"], 8))

Total cost: $ 0.00111975


In [14]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [15]:
print('Q4:', len(chunks))

Q4: 295


In [16]:
documents[:2]

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [17]:
chunks[:2]

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [18]:
index = build_index(chunks)
assistant = RAGBase(index=index, llm_client=openai_client)
question = 'How does the agentic loop keep calling the model until it stops?'
answer = assistant.rag(question)
answer

Response(id='resp_07542c9bee35f11d006a3289b34e0481938ad2955533c3829a', created_at=1781696947.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_07542c9bee35f11d006a3289b5c91881938268cd3356ab5d54', content=[ResponseOutputText(annotations=[], text='It keeps calling the model inside a `while True` loop, and stops when the model returns no `function_call` items in its output.\n\nThe key logic is:\n\n- Call the model.\n- Check `response.output` for any `function_call`.\n- If there is one, run the tool, add the result to `messages`, and loop again.\n- If there are no function calls, `has_function_calls` stays `False`, and the loop `break`s.\n\nSo the stop condition is: **no function calls this turn means the agent is done.**', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, to

In [19]:
print('Q5:', answer.usage.input_tokens)

Q5: 2304


In [20]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [21]:
def search(query, num_results=5):
    boost_dict = {'content': 2.0, }
    #filter_dict = {'course': self.course}

    return index.search(
        query,
        num_results=num_results,
        boost_dict=boost_dict,
        #filter_dict=filter_dict
    )

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [22]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [23]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [24]:
INSTRUCTIONS = '''You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.'''

In [25]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [26]:
prompt = 'How does the agentic loop work, and how is it different from plain RAG?'
result = runner.loop(
    prompt=prompt,
    callback=callback,
)

-> Response received


-> Response received


In [27]:
print('Q6:', )

Q6:


In [28]:
result

LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.", role='developer', phase=None, type=None), EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"agentic loop RAG difference"}', call_id='call_fdiPXDU1GxNNKtwZC8ut0HgA', name='search', type='function_call', id='fc_08565b483bbb655b006a3289b96cf48194bc985fa592241895', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"agentic loop how it works"}', call_id='call_4lmALtLekDC2GTCdwpoVITrW', name='search', type='function_call', id='fc_08565b483bbb655b006a3289b96d0c819481760241ece909da', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"plain RAG vs agentic loop"}', call_id='call_AuFf69pwe4WJTUKHfZix

In [29]:
from openai.types.responses import ResponseFunctionToolCall

tool_calls = [
    msg for msg in result.new_messages
    if isinstance(msg, ResponseFunctionToolCall)
]
tool_calls


[ResponseFunctionToolCall(arguments='{"query":"agentic loop RAG difference"}', call_id='call_fdiPXDU1GxNNKtwZC8ut0HgA', name='search', type='function_call', id='fc_08565b483bbb655b006a3289b96cf48194bc985fa592241895', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"agentic loop how it works"}', call_id='call_4lmALtLekDC2GTCdwpoVITrW', name='search', type='function_call', id='fc_08565b483bbb655b006a3289b96d0c819481760241ece909da', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"plain RAG vs agentic loop"}', call_id='call_AuFf69pwe4WJTUKHfZixT85G', name='search', type='function_call', id='fc_08565b483bbb655b006a3289b96d1881949b2e9445c4145247', namespace=None, status='completed')]

In [30]:
print('Q6:', len(tool_calls))

Q6: 3


In [31]:
import re

def find_q_prints(cells=None):
    if cells is None:
        cells = In

    pattern = re.compile(r"print\(\s*(['\"])(Q\d+):\s*(.*?)\1\s*\)")
    results = []

    for idx, cell in enumerate(cells):
        if not isinstance(cell, str):
            continue

        for match in pattern.finditer(cell):
            q_label = match.group(2)
            message = match.group(3)

            if "'" not in message:
                code = f"print('{q_label}: {message}')"
            elif '"' not in message:
                code = f'print("{q_label}: {message}")'
            else:
                code = f"print({repr(f'{q_label}: {message}')})"

            results.append((idx, q_label, code))

    return results

for idx, q_label, code in find_q_prints():
    print(f"[Cell {idx}] {q_label}  → {code}")

In [ ]:
# end